# Incremental Learning — Pipeline v2 (WINDOW_SIZE=60, corrected threshold)

Same KS-test-triggered SGD fine-tuning mechanism as
`module3_pipeline/incremental_learning.ipynb`, applied to the window=60
model, using the corrected `K_ADAPTIVE` from `adaptive_threshold_blended.ipynb`
(v2) for the per-container decision threshold during streaming.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pickle, joblib, os, time, copy
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import precision_score, recall_score, f1_score
from collections import deque

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
WIN_DIR   = os.path.join(BASE, 'data', 'processed', 'windows_cc1_v2')
MODEL_DIR = os.path.join(BASE, 'models_v2')

WINDOW_SIZE = 60
K_ADAPTIVE     = None   # loaded below, from adaptive_threshold_blended.ipynb's saved result
BUFFER_SIZE    = 500
PRIOR_STRENGTH = None   # loaded below

REFIT_INTERVAL   = 5000
FT_BUFFER_SIZE   = 2000
FT_LR            = 1e-4
FT_EPOCHS        = 5
KS_ALPHA         = 0.001

print('Paths and constants configured.')

Paths and constants configured.


## Step 1 — Validate lightweight/real-time properties + load config from prior steps

In [2]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar); return self.decode(z), mu, logvar
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

meta          = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_meta.pkl'), 'rb'))
static_eval   = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))
adaptive_cfg  = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_adaptive_blended_eval.pkl'), 'rb'))
CLIP          = meta['clip']
GLOBAL_MEAN   = meta['mu_train']
GLOBAL_STD    = meta['sigma_train']
STATIC_FALLBACK = static_eval['thresholds']['val_p99']
K_ADAPTIVE      = adaptive_cfg['k_adaptive']
PRIOR_STRENGTH  = adaptive_cfg['prior_strength']
print(f'Loaded corrected K_ADAPTIVE={K_ADAPTIVE:.4f}  PRIOR_STRENGTH={PRIOR_STRENGTH}')

base_model = VAE(meta['input_dim'], meta['hidden1'], meta['hidden2'], meta['latent_dim'])
base_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'vae_cc1.pt'), map_location='cpu'))
base_model.eval()

n_params = sum(p.numel() for p in base_model.parameters())
file_size_kb = os.path.getsize(os.path.join(MODEL_DIR, 'vae_cc1.pt')) / 1024
x_single = torch.randn(1, meta['input_dim'])
N_TIMING = 2000
with torch.no_grad():
    t0 = time.perf_counter()
    for _ in range(N_TIMING):
        _ = base_model.anomaly_score(x_single)
    t1 = time.perf_counter()
single_latency_ms = (t1 - t0) / N_TIMING * 1000
print(f'Parameters: {n_params:,}   Model size: {file_size_kb:.1f} KB   Single-window latency: {single_latency_ms:.4f} ms')

X_cc1_train = np.clip(np.load(os.path.join(WIN_DIR, 'X_cc1_train.npy')), -CLIP, CLIP).astype(np.float32)
rng = np.random.default_rng(42)
REF_SAMPLE = X_cc1_train[rng.choice(len(X_cc1_train), size=5000, replace=False)]
with torch.no_grad():
    REFERENCE_MSE = base_model.anomaly_score(torch.from_numpy(REF_SAMPLE)).numpy()
print(f'Fixed KS-test reference: {len(REFERENCE_MSE):,} cc1_train windows (original frozen model).')

Loaded corrected K_ADAPTIVE=4.6982  PRIOR_STRENGTH=10000
Parameters: 13,616   Model size: 57.9 KB   Single-window latency: 0.4582 ms
Fixed KS-test reference: 5,000 cc1_train windows (original frozen model).


## Step 2 — Load `drift_cc2` in true global chronological order

In [3]:
FEATURE_COLS = [
    'container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache',
]

def window_meta_with_features(df, feature_cols, window_size=WINDOW_SIZE, stride=1):
    X, cmdb_ids, end_ts, ys, fts = [], [], [], [], []
    for cmdb_id, g in df.sort_values('timestamp').groupby('cmdb_id'):
        data = g[feature_cols].values.astype(np.float32)
        is_gap = g['is_gap'].values
        labels = g['label'].values
        ftypes = g['failure_type'].values.astype(object)
        ts     = g['timestamp'].values
        n = len(g)
        for i in range(0, n - window_size + 1, stride):
            if is_gap[i:i + window_size].any():
                continue
            X.append(data[i:i + window_size])
            cmdb_ids.append(cmdb_id)
            end_ts.append(ts[i + window_size - 1])
            ys.append(int(labels[i:i + window_size].any()))
            w_types = sorted({t for t in ftypes[i:i + window_size] if isinstance(t, str)})
            fts.append(','.join(w_types) if w_types else None)
    return np.stack(X), np.array(cmdb_ids), np.array(end_ts), np.array(ys, dtype=np.int64), np.array(fts, dtype=object)

df_cc2 = pd.read_csv(os.path.join(DATA_DIR, 'drift_complex_case2.csv'), low_memory=False)
df_cc2['is_gap'] = df_cc2['is_gap'].astype(bool)
X_raw, cmdb_ids, end_ts, y_true, ft_true = window_meta_with_features(df_cc2, FEATURE_COLS)

y_saved = np.load(os.path.join(WIN_DIR, 'y_drift_cc2.npy'))
print(f'Rebuilt {len(y_true):,} windows.  Order/label check vs saved y_drift_cc2.npy: {"OK" if np.array_equal(y_true, y_saved) else "MISMATCH!"}')
assert np.array_equal(y_true, y_saved)

pca_bundle = joblib.load(os.path.join(MODEL_DIR, 'cc1_pca.pkl'))
pca = pca_bundle['pca']
X_flat = X_raw.reshape(len(X_raw), -1)
X_pca = np.clip(pca.transform(X_flat), -CLIP, CLIP).astype(np.float32)

order = np.argsort(end_ts, kind='stable')
X_stream   = X_pca[order]
y_stream   = y_true[order]
ft_stream  = ft_true[order]
cmdb_stream = cmdb_ids[order]
ts_stream  = end_ts[order]
print(f'\nStreaming {len(X_stream):,} windows across {len(np.unique(cmdb_stream))} containers.')

Rebuilt 76,167 windows.  Order/label check vs saved y_drift_cc2.npy: OK

Streaming 76,167 windows across 27 containers.


## Step 3 — Streaming simulator (control vs. treatment)

In [4]:
def run_stream(model, X_stream, cmdb_stream, incremental_learning, reference_mse, verbose=True):
    model = copy.deepcopy(model)
    opt = torch.optim.Adam(model.parameters(), lr=FT_LR)

    preds = np.zeros(len(X_stream), dtype=np.int64)
    mse_trace = np.zeros(len(X_stream), dtype=np.float64)
    threshold_buffers = {}
    ft_pool = deque(maxlen=FT_BUFFER_SIZE)
    n_finetunes = 0
    finetune_events = []

    for i in range(len(X_stream)):
        x_i = X_stream[i]
        cid = cmdb_stream[i]
        x_t = torch.from_numpy(x_i).unsqueeze(0)

        with torch.no_grad():
            mse = model.anomaly_score(x_t).item()
        mse_trace[i] = mse

        buf = threshold_buffers.setdefault(cid, deque(maxlen=BUFFER_SIZE))
        n_local = len(buf)
        w = n_local / (n_local + PRIOR_STRENGTH)
        if n_local == 0:
            lm, ls = GLOBAL_MEAN, GLOBAL_STD
        else:
            arr = np.fromiter(buf, dtype=np.float64); lm, ls = arr.mean(), arr.std()
        t = (w * lm + (1 - w) * GLOBAL_MEAN) + K_ADAPTIVE * (w * ls + (1 - w) * GLOBAL_STD)

        is_anom = mse > t
        preds[i] = int(is_anom)
        if not is_anom:
            buf.append(mse)
            ft_pool.append(x_i)

        if incremental_learning and (i + 1) % REFIT_INTERVAL == 0 and len(ft_pool) >= FT_BUFFER_SIZE // 2:
            with torch.no_grad():
                recent_mse = model.anomaly_score(torch.from_numpy(np.stack(list(ft_pool)))).numpy()
            ks_stat, p_value = stats.ks_2samp(reference_mse, recent_mse)
            drifted = p_value < KS_ALPHA
            if drifted:
                Xb = torch.from_numpy(np.stack(list(ft_pool)))
                model.train()
                for _ in range(FT_EPOCHS):
                    opt.zero_grad()
                    recon, mu, logvar = model(Xb)
                    recon_loss = nn.functional.mse_loss(recon, Xb, reduction='mean')
                    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
                    loss = recon_loss + meta['beta_max'] * kl
                    loss.backward()
                    opt.step()
                model.eval()
                n_finetunes += 1
                finetune_events.append(i)
                if verbose:
                    print(f'  [window {i+1:>6}] drift detected (KS p={p_value:.2e}) -> fine-tuned')

    return preds, mse_trace, n_finetunes, finetune_events

print('Streaming simulator defined.')

Streaming simulator defined.


## Step 4 — Run control (no incremental learning) vs. treatment (incremental learning ON)

In [5]:
print('Running CONTROL (incremental learning OFF) ...')
preds_control, mse_control, nft_control, _ = run_stream(base_model, X_stream, cmdb_stream, incremental_learning=False, reference_mse=REFERENCE_MSE, verbose=False)
print(f'  done. n_finetunes={nft_control} (should be 0)')

print('\nRunning TREATMENT (incremental learning ON) ...')
preds_treat, mse_treat, nft_treat, finetune_events = run_stream(base_model, X_stream, cmdb_stream, incremental_learning=True, reference_mse=REFERENCE_MSE, verbose=True)
print(f'  done. n_finetunes={nft_treat}')

Running CONTROL (incremental learning OFF) ...
  done. n_finetunes=0 (should be 0)

Running TREATMENT (incremental learning ON) ...
  [window   5000] drift detected (KS p=4.26e-321) -> fine-tuned
  [window  10000] drift detected (KS p=3.24e-321) -> fine-tuned
  [window  15000] drift detected (KS p=4.16e-321) -> fine-tuned
  [window  20000] drift detected (KS p=6.85e-319) -> fine-tuned
  [window  25000] drift detected (KS p=3.73e-321) -> fine-tuned
  [window  30000] drift detected (KS p=2.09e-317) -> fine-tuned
  [window  35000] drift detected (KS p=8.41e-304) -> fine-tuned
  [window  40000] drift detected (KS p=4.16e-321) -> fine-tuned
  [window  45000] drift detected (KS p=3.66e-312) -> fine-tuned
  [window  50000] drift detected (KS p=5.37e-276) -> fine-tuned
  [window  55000] drift detected (KS p=2.66e-257) -> fine-tuned
  [window  60000] drift detected (KS p=3.18e-302) -> fine-tuned
  [window  65000] drift detected (KS p=2.77e-290) -> fine-tuned
  [window  70000] drift detected (KS

## Step 5 — Compare: control vs. treatment

In [6]:
def report(name, preds):
    p = precision_score(y_stream, preds, zero_division=0)
    r = recall_score(y_stream, preds, zero_division=0)
    f1 = f1_score(y_stream, preds, zero_division=0)
    print(f'{name:12s} precision={p:.3f}  recall={r:.3f}  F1={f1:.3f}')
    return {'precision': p, 'recall': r, 'f1': f1}

print('=== drift_cc2 (v2, window=60): control vs. treatment ===')
result_control = report('control', preds_control)
result_treat   = report('treatment', preds_treat)
print(f'\nF1 change from incremental learning: {result_treat["f1"] - result_control["f1"]:+.3f}')

=== drift_cc2 (v2, window=60): control vs. treatment ===
control      precision=0.089  recall=0.806  F1=0.159
treatment    precision=0.092  recall=0.800  F1=0.165

F1 change from incremental learning: +0.006


## Step 6 — Save

In [7]:
save_results = {
    'lightweight_validation': {'n_params': n_params, 'file_size_kb': file_size_kb, 'single_window_latency_ms': single_latency_ms},
    'incremental_learning': {
        'refit_interval': REFIT_INTERVAL, 'ft_buffer_size': FT_BUFFER_SIZE, 'ft_lr': FT_LR, 'ft_epochs': FT_EPOCHS,
        'n_finetunes': nft_treat, 'finetune_events': finetune_events,
        'control': result_control, 'treatment': result_treat,
    },
}
out_path = os.path.join(MODEL_DIR, 'incremental_learning_eval.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\models_v2\incremental_learning_eval.pkl
